In [10]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
import yfinance as yf

os.chdir('E:/Mutual Fund Analytics')

nav = pd.read_csv('data/processed/clean_nav_history.csv')
perf = pd.read_csv('data/processed/clean_schema_performance.csv')

# Parsing date
nav['date'] = pd.to_datetime(nav['date'])
nav = nav.sort_values(['amfi_code', 'date'])

os.makedirs('charts', exist_ok = True)

print("Loaded!")
print("NAV rows:", len(nav))
print("Funds:", nav['amfi_code'].nunique())

Loaded!
NAV rows: 71960
Funds: 40


In [11]:
# Task 1 - Daily Returns
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

# Validate distribution
print(nav['daily_return'].describe())

# Plot distribution
fig = px.histogram(
    nav.dropna(),
    x='daily_return',
    nbins=100,
    title='Distribution of Daily Returns - All Funds'
)
fig.show()

count    71920.000000
mean         0.001231
std          0.040326
min         -0.523526
25%          0.000000
50%          0.000000
75%          0.000000
max          0.436832
Name: daily_return, dtype: float64


In [12]:
# Task 2 - CAGR
def calculate_cagr(group, years):
    group = group.sort_values('date')
    end_date = group['date'].max()
    start_date = end_date - pd.DateOffset(years=years)
    filtered = group[group['date'] >= start_date]
    if len(filtered) < 2:
        return None
    nav_start = filtered.iloc[0]['nav']
    nav_end = filtered.iloc[-1]['nav']
    return (nav_end / nav_start) ** (1 / years) - 1

cagr_results = []
for code, group in nav.groupby('amfi_code'):
    cagr_results.append({
        'amfi_code': code,
        'cagr_1yr': calculate_cagr(group, 1),
        'cagr_3yr': calculate_cagr(group, 3),
        'cagr_5yr': calculate_cagr(group, 5)
    })

cagr_df = pd.DataFrame(cagr_results)
print(cagr_df.head())

   amfi_code  cagr_1yr  cagr_3yr  cagr_5yr
0     100016 -0.050140  0.013412  0.024722
1     100025  0.044768  0.035574  0.039329
2     100033  0.551274  0.321169  0.264744
3     101206  0.481654  0.284268  0.199172
4     101207 -0.300416 -0.027768  0.055929


In [13]:
# Task 3 - Sharpe Ratio
rf_daily = 0.065 / 252  # RBI repo rate daily

sharpe_results = []
for code, group in nav.groupby('amfi_code'):
    returns = group['daily_return'].dropna()
    excess_returns = returns - rf_daily
    sharpe = (excess_returns.mean() / returns.std()) * np.sqrt(252)
    sharpe_results.append({'amfi_code': code, 'sharpe_ratio': sharpe})

sharpe_df = pd.DataFrame(sharpe_results).sort_values('sharpe_ratio', ascending=False)
print(sharpe_df)

    amfi_code  sharpe_ratio
39     149324      0.701514
21     119598      0.610974
30     120843      0.600055
25     120505      0.587454
34     148567      0.583626
38     149323      0.562143
16     119094      0.560176
2      100033      0.519991
36     148569      0.513461
19     119551      0.498546
11     118634      0.462001
9      118632      0.447396
35     148568      0.445837
24     120504      0.440361
3      101206      0.422651
8      102887      0.420214
32     125497      0.402322
22     119599      0.390032
23     120503      0.385711
20     119552      0.373115
17     119095      0.361058
4      101207      0.349911
12     118635      0.344942
26     120506      0.338516
37     149322      0.333760
28     120841      0.327278
6      102885      0.322773
10     118633      0.317952
33     125498      0.301285
15     119093      0.236558
7      102886      0.206679
29     120842      0.174714
0      100016      0.142861
14     119092      0.132666
13     118636     -0

In [14]:
# Task 4 - Sortino Ratio
sortino_results = []
for code, group in nav.groupby('amfi_code'):
    returns = group['daily_return'].dropna()
    excess_returns = returns - rf_daily
    downside = returns[returns < 0].std()
    sortino = (excess_returns.mean() / downside) * np.sqrt(252)
    sortino_results.append({'amfi_code': code, 'sortino_ratio': sortino})

sortino_df = pd.DataFrame(sortino_results).sort_values('sortino_ratio', ascending=False)
print(sortino_df)

    amfi_code  sortino_ratio
21     119598       0.430052
39     149324       0.425362
11     118634       0.355517
30     120843       0.327032
38     149323       0.322678
22     119599       0.318077
25     120505       0.315413
36     148569       0.307605
16     119094       0.302815
2      100033       0.280002
17     119095       0.273092
4      101207       0.251922
3      101206       0.246029
34     148567       0.244094
35     148568       0.242963
19     119551       0.239684
24     120504       0.238624
9      118632       0.229879
32     125497       0.223849
8      102887       0.223315
33     125498       0.215461
20     119552       0.201496
26     120506       0.201273
37     149322       0.187202
10     118633       0.185329
23     120503       0.182140
28     120841       0.177762
7      102886       0.176892
12     118635       0.164267
6      102885       0.163669
15     119093       0.152744
29     120842       0.134135
0      100016       0.097436
14     119092 

In [16]:
# Fetch Nifty 100
nifty = yf.download('^CNX100', start='2022-01-01', end='2026-01-01')
nifty = nifty['Close'].pct_change().dropna()
nifty = nifty.reset_index()
nifty.columns = ['date', 'nifty_return']
nifty['date'] = pd.to_datetime(nifty['date'])
nifty = nifty.set_index('date')

alpha_beta_results = []
for code, group in nav.groupby('amfi_code'):
    group = group.set_index('date')[['daily_return']].dropna()
    merged = group.join(nifty, how='inner')
    if len(merged) < 30:
        continue
    slope, intercept, r, p, se = stats.linregress(merged['nifty_return'], merged['daily_return'])
    alpha_beta_results.append({
        'amfi_code': code,
        'alpha': intercept * 252,
        'beta': slope
    })

alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_df.to_csv('data/processed/alpha_beta.csv', index=False)
print(alpha_beta_df)

[*********************100%***********************]  1 of 1 completed

    amfi_code     alpha      beta
0      100016  0.259035  0.116435
1      100025  0.054047 -0.018706
2      100033  0.526101 -0.089472
3      101206  0.424848 -0.013083
4      101207  0.281748  0.078260
5      101208  0.062675 -0.030853
6      102885  0.321367 -0.048702
7      102886  0.128453 -0.199609
8      102887  0.633411 -0.088265
9      118632  0.274081 -0.055696
10     118633  0.288422 -0.068729
11     118634  0.513323  0.008444
12     118635  0.279945  0.032730
13     118636  0.025802 -0.083330
14     119092  0.156747 -0.075536
15     119093  0.371527  0.035195
16     119094  0.718546 -0.137961
17     119095  0.415365  0.109850
18     119120  0.051894  0.011755
19     119551  0.605948  0.079292
20     119552  0.350630  0.113811
21     119598  0.666629 -0.237363
22     119599  0.432955 -0.013627
23     120503  0.375849 -0.087511
24     120504  0.448142  0.105799
25     120505  0.688765 -0.259724
26     120506  0.333276 -0.047566
27     120507  0.080305 -0.037134
28     120841 

In [7]:
# Task 6 - Maximum Drawdown
drawdown_results = []
for code, group in nav.groupby('amfi_code'):
    group = group.sort_values('date')
    group['running_max'] = group['nav'].cummax()
    group['drawdown'] = group['nav'] / group['running_max'] - 1
    max_dd = group['drawdown'].min()
    max_dd_date = group.loc[group['drawdown'].idxmin(), 'date']
    drawdown_results.append({
        'amfi_code': code,
        'max_drawdown': max_dd,
        'max_drawdown_date': max_dd_date
    })

drawdown_df = pd.DataFrame(drawdown_results).sort_values('max_drawdown')
print(drawdown_df)

    amfi_code  max_drawdown max_drawdown_date
39     149324     -0.523526        2025-03-01
17     119095     -0.498422        2026-11-05
22     119599     -0.497372        2025-10-10
21     119598     -0.433908        2023-10-01
38     149323     -0.428700        2023-10-02
34     148567     -0.408232        2024-11-01
16     119094     -0.381816        2024-12-02
25     120505     -0.377792        2022-04-01
19     119551     -0.372071        2025-10-01
30     120843     -0.365886        2024-05-02
4      101207     -0.355330        2022-02-06
11     118634     -0.339884        2022-04-02
9      118632     -0.334617        2023-09-01
2      100033     -0.334540        2024-11-01
8      102887     -0.326393        2025-03-01
35     148568     -0.324661        2022-07-03
28     120841     -0.318009        2023-02-02
26     120506     -0.304018        2025-10-01
3      101206     -0.298823        2026-02-01
23     120503     -0.298281        2022-03-01
20     119552     -0.294361       

In [17]:
# Task 7 - Fund Scorecard
scorecard = cagr_df.merge(sharpe_df, on='amfi_code')
scorecard = scorecard.merge(sortino_df, on='amfi_code')
scorecard = scorecard.merge(alpha_beta_df, on='amfi_code')
scorecard = scorecard.merge(drawdown_df, on='amfi_code')
scorecard = scorecard.merge(perf[['amfi_code', 'expense_ratio_pct']], on='amfi_code')

# Rank each metric
scorecard['rank_3yr']     = scorecard['cagr_3yr'].rank(ascending=True)
scorecard['rank_sharpe']  = scorecard['sharpe_ratio'].rank(ascending=True)
scorecard['rank_alpha']   = scorecard['alpha'].rank(ascending=True)
scorecard['rank_expense'] = scorecard['expense_ratio_pct'].rank(ascending=False)
scorecard['rank_maxdd']   = scorecard['max_drawdown'].rank(ascending=False)

# Composite score
scorecard['score'] = (
    0.30 * scorecard['rank_3yr'] +
    0.25 * scorecard['rank_sharpe'] +
    0.20 * scorecard['rank_alpha'] +
    0.15 * scorecard['rank_expense'] +
    0.10 * scorecard['rank_maxdd']
)

# Normalize to 0-100
scorecard['score'] = (scorecard['score'] / scorecard['score'].max()) * 100
scorecard = scorecard.sort_values('score', ascending=False)
scorecard.to_csv('data/processed/fund_scorecard.csv', index=False)
print(scorecard[['amfi_code', 'score']].head(10))

    amfi_code       score
25     120505  100.000000
34     148567   99.009901
39     149324   98.939180
16     119094   97.029703
30     120843   96.605375
21     119598   91.796322
2      100033   87.835926
19     119551   80.975955
24     120504   80.056577
38     149323   79.207921


In [18]:
# Task 8 - Benchmark Comparison
top5 = scorecard.head(5)['amfi_code'].tolist()
nav_top5 = nav[nav['amfi_code'].isin(top5)]

fig = px.line(title='Top 5 Funds Cumulative Returns 2022-2026')

for code in top5:
    fund_data = nav_top5[nav_top5['amfi_code'] == code].copy()
    fund_data['cumulative'] = fund_data['daily_return'].cumsum()
    fig.add_scatter(x=fund_data['date'], y=fund_data['cumulative'], name=str(code))

fig.show()